# Significancia estadística y tamaño de efecto en clasificación de ratones

Este notebook reemplaza el análisis original por un flujo reproducible para muestras pequeñas y múltiples estímulos.

## Preguntas que responde

1. **Por prueba individual:** ¿la exactitud balanceada del SVM supera lo esperable por azar para un estímulo y una métrica concretos?
2. **Por metodología:** ¿una métrica (energía, entropía, KL, coactivación o Hurst) separa los grupos en al menos un estímulo, o de forma consistente entre estímulos?
3. **Selección de estímulos:** ¿qué estímulos siguen siendo relevantes después de corregir la multiplicidad?
4. **Magnitud del efecto:** ¿cuán separadas están las distribuciones mediante Cohen *d*, Hedges *g* y medidas complementarias?

La prueba principal es una **permutación de etiquetas que repite completamente la validación cruzada**. Para respetar el diseño factorial 2×2, la edad se permuta dentro de cada genotipo y el genotipo dentro de cada edad. Las permutaciones se sincronizan entre estímulos de una misma metodología, lo que permite una corrección **maxT** y pruebas globales de la metodología.

> Referencia metodológica: Combrisson & Jerbi (2015), *Exceeding chance level by chance: The caveat of theoretical chance levels in brain signal classification and statistical assessment of decoding accuracy*.

> **Identificación de sujetos:** en estas tablas, el identificador del ratón está almacenado en el índice (`MR-0644`, `MR-0645`, etc.). El notebook conserva ese índice y lo copia explícitamente a `subject_id`. La columna `type` se usa solo para inferir genotipo y edad.


In [ ]:
from __future__ import annotations

from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Callable, Iterable, Mapping, Sequence
import json
import math
import re
import warnings

import joblib
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from scipy.stats import binom
from sklearn.base import clone
from sklearn.covariance import LedoitWolf
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, balanced_accuracy_score
from sklearn.model_selection import LeaveOneOut, StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings('once')
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 180)

# -----------------------------------------------------------------------------
# CONFIGURACIÓN PRINCIPAL
# -----------------------------------------------------------------------------
PROJECT_ROOT = Path('.')
OUTPUT_DIR = PROJECT_ROOT / 'statistical_results'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Para un análisis final se recomiendan 9_999 o más permutaciones. Durante
# desarrollo puede usarse 99/499/999. El p mínimo posible es 1/(B+1).
N_PERMUTATIONS = 9_999
N_JOBS = -1
RANDOM_STATE = 20260713
ALPHA = 0.05

# 'loo' reproduce el esquema leave-one-out. El paper advierte que LOO puede
# presentar más varianza que k-fold en muestras pequeñas. Como sensibilidad,
# puede cambiarse a 'stratified_kfold'.
CV_SCHEME = 'loo'                 # 'loo' | 'stratified_kfold'
N_SPLITS = 5                      # solo para stratified_kfold

# Modelo inferencial recomendado: hiperparámetros fijados a priori. Si se usan
# modelos guardados, se clona el estimador para que se vuelva a ajustar dentro
# de cada fold; no se reutiliza su estado entrenado.
MODEL_POLICY = 'fixed'            # 'fixed' | 'saved_template'
SVM_KERNEL = 'linear'
SVM_C = 1.0
SVM_CLASS_WEIGHT = 'balanced'

# Las comparaciones simples son útiles para explorar una interacción edad ×
# genotipo, pero reducen aún más n. Se dejan desactivadas por defecto.
INCLUDE_SIMPLE_EFFECTS = False

# Bootstrap descriptivo para intervalos de tamaño de efecto.
N_BOOTSTRAP_EFFECT = 2_000
SAVE_NULL_DISTRIBUTIONS = False

# Filtros opcionales para ejecutar subconjuntos y reducir tiempo de cómputo.
# Use None para incluir todo, o un set de nombres.
SELECTED_STIMULUS_TYPES: set[str] | None = None
SELECTED_ANALYSES: set[str] | None = None
SELECTED_CONTRASTS: set[str] | None = None

# Si una tabla 2D de Hurst no tiene nombres de columnas inequívocos para Hx/Hy,
# especifique aquí los grupos manualmente. Ejemplo:
# FEATURE_GROUP_OVERRIDES[("2D_stochastic", "hurst")] = {
#     "stimulus_0": ["H_x_0", "H_y_0"],
#     "stimulus_1": ["H_x_1", "H_y_1"],
#     "stimulus_2": ["H_x_2", "H_y_2"],
# }
FEATURE_GROUP_OVERRIDES: dict[tuple[str, str], dict[str, list[str]]] = {}

# Mapeo opcional explícito de modelos guardados por prueba.
# key = (stimulus_type, analysis, contrast, stimulus_id)
SAVED_MODEL_OVERRIDES: dict[tuple[str, str, str, str], str] = {}


## 1. Diseño experimental y familias de hipótesis

La corrección de multiplicidad no debe definirse solo porque los archivos tengan una estructura parecida. La familia principal se define por una **pregunta biológica común**:

- tipo de estímulo (`1D_deterministic`, `1D_stochastic`, `2D_stochastic`),
- contraste (`age_main` o `condition_main`),
- esquema de validación cruzada.

Dentro de esa familia se aplica FDR a todas las combinaciones metodología–estímulo. Adicionalmente, dentro de cada metodología se usa maxT, que conserva la dependencia generada por utilizar los mismos ratones y corrige explícitamente la selección del mejor estímulo.

Los contrastes principales son:

- `age_main`: 3 meses vs. 6 meses, permutando edad **dentro de genotipo**.
- `condition_main`: Wild-Type vs. 5xFAD, permutando genotipo **dentro de edad**.

Esto prueba efectos principales controlando el otro factor. Las comparaciones dentro de cada estrato pueden habilitarse para estudiar posibles interacciones.


In [ ]:
# -----------------------------------------------------------------------------
# ESTRUCTURAS DE CONFIGURACIÓN
# -----------------------------------------------------------------------------
@dataclass(frozen=True)
class StimulusSetConfig:
    stimulus_type: str
    base_dir: Path
    model_dir: Path
    expected_stimuli: int
    files: Mapping[str, str]


@dataclass(frozen=True)
class Contrast:
    name: str
    target: str
    positive_class: str
    negative_class: str
    nuisance: str | None = None
    filters: Mapping[str, str] | None = None


@dataclass(frozen=True)
class FeatureGroup:
    stimulus_id: str
    columns: tuple[str, ...]


STIMULUS_SETS = [
    StimulusSetConfig(
        stimulus_type='1D_deterministic',
        base_dir=PROJECT_ROOT / '1D_ind_sin_analyses',
        model_dir=PROJECT_ROOT / '1D_ind_sin_analyses' / 'models',
        expected_stimuli=21,
        files={
            'energy': 'Energy_level3.json',
            'coactivation': 'Coactiv.json',
            'entropy': 'Entropy.json',
            'kl_divergence': 'kl_div.json',
        },
    ),
    StimulusSetConfig(
        stimulus_type='1D_stochastic',
        base_dir=PROJECT_ROOT / '1D_brownian_analyses',
        model_dir=PROJECT_ROOT / '1D_brownian_analyses' / 'models',
        expected_stimuli=3,
        files={
            'energy': 'Energy_level3.json',
            'coactivation': 'Coactiv.json',
            'entropy': 'Entropy.json',
            'kl_divergence': 'kldiv.json',
            'hurst': 'Hurst_regression.json',
        },
    ),
    StimulusSetConfig(
        stimulus_type='2D_stochastic',
        base_dir=PROJECT_ROOT / '2D_analyses',
        model_dir=PROJECT_ROOT / '2D_analyses' / 'models',
        expected_stimuli=3,
        files={
            'energy': 'Energy_level3.json',
            'coactivation': 'Coactiv.json',
            'entropy': 'Entropy.json',
            'kl_divergence': 'kldiv.json',
            'hurst': 'h_est.json',
        },
    ),
]


def build_contrasts(include_simple_effects: bool = False) -> list[Contrast]:
    contrasts = [
        Contrast(
            name='age_main', target='age', nuisance='condition',
            negative_class='3m', positive_class='6m',
        ),
        Contrast(
            name='condition_main', target='condition', nuisance='age',
            negative_class='Wild-Type', positive_class='5xFAD',
        ),
    ]
    if include_simple_effects:
        contrasts.extend([
            Contrast('condition_at_3m', 'condition', '5xFAD', 'Wild-Type', filters={'age': '3m'}),
            Contrast('condition_at_6m', 'condition', '5xFAD', 'Wild-Type', filters={'age': '6m'}),
            Contrast('age_within_WT', 'age', '6m', '3m', filters={'condition': 'Wild-Type'}),
            Contrast('age_within_5xFAD', 'age', '6m', '3m', filters={'condition': '5xFAD'}),
        ])
    return contrasts


CONTRASTS = build_contrasts(INCLUDE_SIMPLE_EFFECTS)


### Identificación de sujetos en JSON
El lector acepta tanto IDs almacenados en el índice como IDs recuperados en la columna `sample_id`. Esta última se excluye de las variables predictoras para evitar que el identificador entre al SVM.


In [ ]:
# -----------------------------------------------------------------------------
# IDENTIFICACIÓN SEGURA DE LOS RATONES Y ETIQUETAS
# -----------------------------------------------------------------------------
AGE_PATTERNS = {
    '3m': re.compile(r'(?i)(?:^|[^0-9])3\s*(?:m|mo|month|months|mes|meses)(?:[^0-9]|$)'),
    '6m': re.compile(r'(?i)(?:^|[^0-9])6\s*(?:m|mo|month|months|mes|meses)(?:[^0-9]|$)'),
}
CONDITION_PATTERNS = {
    '5xFAD': re.compile(r'(?i)5\s*x\s*FAD'),
    'Wild-Type': re.compile(r'(?i)(?:wild[\s_-]*type|(?:^|[^A-Za-z])WT(?:[^A-Za-z]|$))'),
}

# Se usan solo como respaldo. En los archivos actuales, el ID principal está
# almacenado en el índice del DataFrame.
SUBJECT_ID_CANDIDATES = (
    'subject_id', 'sample_id', 'mouse_id', 'animal_id', 'raton_id', 'ratón_id',
    'mouse', 'animal', 'subject', 'id',
)
GROUP_LABEL_CANDIDATES = ('type', 'group', 'label', 'class')


def _match_unique(text: str, patterns: Mapping[str, re.Pattern], variable: str) -> str:
    text = str(text)
    matches = [label for label, pattern in patterns.items() if pattern.search(text)]
    if len(matches) != 1:
        raise ValueError(
            f'No se pudo inferir {variable} de {text!r}. Coincidencias: {matches}. '
            'Ajuste los patrones o agregue columnas explícitas age/condition.'
        )
    return matches[0]


def _is_complete_unique(series: pd.Series) -> bool:
    values = series.astype('string')
    return bool(values.notna().all() and values.str.strip().ne('').all() and values.is_unique)


def _is_default_positional_index(index: pd.Index) -> bool:
    """Detecta índices 0, 1, ..., n-1 que no representan IDs experimentales."""
    if isinstance(index, pd.RangeIndex):
        return index.start == 0 and index.step == 1 and index.stop == len(index)
    try:
        numeric = pd.to_numeric(pd.Index(index), errors='raise')
        return np.array_equal(np.asarray(numeric), np.arange(len(index)))
    except (TypeError, ValueError):
        return False


def _subject_ids_from_index(df: pd.DataFrame) -> pd.Series | None:
    """Devuelve los IDs del índice cuando este contiene IDs reales y únicos."""
    if not df.index.is_unique or _is_default_positional_index(df.index):
        return None

    index_values = pd.Series(
        df.index.map(str),
        index=df.index,
        name='subject_id',
        dtype='string',
    )
    if not _is_complete_unique(index_values):
        return None
    return index_values


def _select_label_source(out: pd.DataFrame) -> pd.Series:
    """Texto usado para inferir edad y genotipo, nunca para identificar al ratón."""
    for column in GROUP_LABEL_CANDIDATES:
        if column in out.columns and out[column].notna().all():
            return out[column].astype(str)
    return out['subject_id'].astype(str)


def _normalize_age_value(value: object) -> str:
    text = str(value).strip()
    aliases = {
        'young': '3m', 'joven': '3m', '3': '3m', '3.0': '3m',
        'old': '6m', 'viejo': '6m', '6': '6m', '6.0': '6m',
    }
    lowered = text.lower()
    if lowered in aliases:
        return aliases[lowered]
    if text in {'3m', '6m'}:
        return text
    return _match_unique(text, AGE_PATTERNS, 'age')


def _normalize_condition_value(value: object) -> str:
    text = str(value).strip()
    compact = re.sub(r'[\s_-]+', '', text).lower()
    aliases = {
        'wt': 'Wild-Type', 'wildtype': 'Wild-Type',
        '5xfad': '5xFAD',
    }
    if compact in aliases:
        return aliases[compact]
    return _match_unique(text, CONDITION_PATTERNS, 'condition')


def add_subject_metadata(df: pd.DataFrame) -> pd.DataFrame:
    """
    Agrega ``subject_id``, ``age`` y ``condition``.

    Para la estructura actual de los datos, el identificador del ratón se toma
    primero del índice (por ejemplo, ``MR-0644``). La columna ``type`` contiene
    la etiqueta grupal (por ejemplo, ``WT_3m``) y se utiliza exclusivamente para
    inferir edad y condición.

    Prioridad para ``subject_id``:
      1. índice original, si es no posicional, completo y único;
      2. columna explícita completa y única (sample_id, mouse_id, animal_id, etc.);
      3. error explícito: no se generan IDs sintéticos silenciosamente.
    """
    out = df.copy()

    # Caso esperado para estos archivos: MR-0644, MR-0645, ... en el índice.
    index_ids = _subject_ids_from_index(out)
    if index_ids is not None:
        # Se usa to_numpy para asignar por posición y evitar cualquier alineación
        # inesperada entre índices al crear la nueva columna.
        out['subject_id'] = index_ids.to_numpy(dtype=str)
        id_source = 'índice original'
    else:
        id_source = None
        for column in SUBJECT_ID_CANDIDATES:
            if column in out.columns and _is_complete_unique(out[column]):
                out['subject_id'] = out[column].astype(str).to_numpy()
                id_source = f'columna {column!r}'
                break

    if id_source is None:
        raise ValueError(
            'No se encontró un identificador único por ratón. Se esperaba un '
            'índice como MR-0644 o una columna explícita sample_id/mouse_id/subject_id. '
            f'Índice observado: {out.index[:5].tolist()}; '
            f'columnas: {out.columns.tolist()}.'
        )

    # Verificación temprana: un ratón debe aparecer una sola vez por tabla.
    duplicated = out.loc[out['subject_id'].duplicated(keep=False), 'subject_id'].tolist()
    if duplicated:
        raise ValueError(
            'Se esperan estadísticas resumidas con una fila por ratón. '
            f'Se encontraron subject_id duplicados: {duplicated[:10]}'
        )

    # Edad y condición se toman de columnas explícitas cuando existen; en caso
    # contrario, se infieren desde type (WT_3m, 5xFAD_6m, etc.).
    label_source = _select_label_source(out)

    if 'age' in out.columns:
        out['age'] = out['age'].map(_normalize_age_value)
    else:
        out['age'] = label_source.map(lambda value: _match_unique(value, AGE_PATTERNS, 'age'))

    condition_column = next(
        (column for column in ('condition', 'genotype') if column in out.columns),
        None,
    )
    if condition_column is not None:
        out['condition'] = out[condition_column].map(_normalize_condition_value)
    else:
        out['condition'] = label_source.map(
            lambda value: _match_unique(value, CONDITION_PATTERNS, 'condition')
        )

    out.attrs['subject_id_source'] = id_source
    return out


In [ ]:
# -----------------------------------------------------------------------------
# CARGA DE TABLAS Y DESCUBRIMIENTO DE ESTÍMULOS/COLUMNAS
# -----------------------------------------------------------------------------
META_COLUMNS = {
    'subject_id', 'sample_id', 'mouse_id', 'animal_id', 'raton_id', 'ratón_id',
    'mouse', 'animal', 'subject', 'id',
    'type', 'group', 'class', 'age', 'condition', 'genotype', 'label',
}


def read_feature_table(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(path)

    # Dependiendo de cómo se guardó el JSON, el ID puede reaparecer como
    # índice real o como una columna explícita (habitualmente sample_id).
    # add_subject_metadata acepta ambas representaciones sin generar IDs.
    df = pd.read_json(path)
    df.columns = [str(column) for column in df.columns]

    result = add_subject_metadata(df)
    if result.attrs.get('subject_id_source') != 'índice original':
        warnings.warn(
            f'{path.name}: el subject_id no provino del índice. '
            f'Fuente utilizada: {result.attrs.get("subject_id_source")}.'
        )
    return result


def _axis_and_stimulus(column: str) -> tuple[str, str] | None:
    s = str(column).strip()
    patterns = [
        r'(?i)^(?:H[_\-\s]*)?(?P<axis>x|y)[_\-\s]*(?P<stim>.+)$',
        r'(?i)^(?P<stim>.+?)[_\-\s]*(?:H[_\-\s]*)?(?P<axis>x|y)$',
    ]
    for pattern in patterns:
        match = re.match(pattern, s)
        if match:
            return match.group('axis').lower(), match.group('stim').strip('_- ')
    return None


def discover_feature_groups(
    df: pd.DataFrame,
    stimulus_type: str,
    analysis: str,
    expected_stimuli: int,
) -> list[FeatureGroup]:
    override = FEATURE_GROUP_OVERRIDES.get((stimulus_type, analysis))
    if override:
        missing = [c for cols in override.values() for c in cols if c not in df.columns]
        if missing:
            raise KeyError(f'Columnas del override ausentes: {missing}')
        return [FeatureGroup(str(stim), tuple(map(str, cols))) for stim, cols in override.items()]

    feature_columns = [str(c) for c in df.columns if str(c) not in META_COLUMNS]
    if not feature_columns:
        raise ValueError(f'No se encontraron columnas de features para {stimulus_type}/{analysis}.')

    # Para Hurst 2D se intenta identificar pares Hx/Hy por nombre.
    if stimulus_type == '2D_stochastic' and analysis == 'hurst':
        parsed = {column: _axis_and_stimulus(column) for column in feature_columns}
        if all(value is not None for value in parsed.values()):
            grouped: dict[str, dict[str, str]] = {}
            for column, (axis, stim) in parsed.items():
                grouped.setdefault(stim, {})[axis] = column
            if all(set(axis_map) == {'x', 'y'} for axis_map in grouped.values()):
                groups = [
                    FeatureGroup(str(stim), (axis_map['x'], axis_map['y']))
                    for stim, axis_map in sorted(grouped.items(), key=lambda item: str(item[0]))
                ]
                if len(groups) != expected_stimuli:
                    warnings.warn(
                        f'{stimulus_type}/{analysis}: se esperaban {expected_stimuli} estímulos '
                        f'y se detectaron {len(groups)} pares Hx/Hy.'
                    )
                return groups

        raise ValueError(
            'No fue posible agrupar automáticamente Hx/Hy en 2D. Defina '
            'FEATURE_GROUP_OVERRIDES[("2D_stochastic", "hurst")].'
        )

    # En las demás tablas cada columna se interpreta como un estímulo escalar.
    groups = [FeatureGroup(str(column), (str(column),)) for column in feature_columns]
    if len(groups) != expected_stimuli:
        warnings.warn(
            f'{stimulus_type}/{analysis}: se esperaban {expected_stimuli} estímulos '
            f'y se detectaron {len(groups)} columnas. Revise la tabla/configuración.'
        )
    return groups


def validate_project_files(configs: Sequence[StimulusSetConfig]) -> pd.DataFrame:
    """Comprueba existencia, lectura e identificación de sujetos en cada tabla.

    La función no modifica los datos. Además de verificar la ruta, intenta
    cargar cada JSON mediante ``read_feature_table`` para detectar de forma
    temprana problemas con el índice, IDs duplicados o etiquetas de grupo.
    """
    rows: list[dict[str, object]] = []

    for cfg in configs:
        for analysis, filename in cfg.files.items():
            path = cfg.base_dir / filename
            row: dict[str, object] = {
                'stimulus_type': cfg.stimulus_type,
                'analysis': analysis,
                'path': str(path),
                'exists': path.exists(),
                'readable': False,
                'n_rows': np.nan,
                'n_subjects': np.nan,
                'subject_id_source': None,
                'error': None,
            }

            if not path.exists():
                row['error'] = 'Archivo no encontrado'
                rows.append(row)
                continue

            try:
                df = read_feature_table(path)
                row.update({
                    'readable': True,
                    'n_rows': int(len(df)),
                    'n_subjects': int(df['subject_id'].nunique()),
                    'subject_id_source': df.attrs.get('subject_id_source'),
                })
            except Exception as exc:
                row['error'] = f'{type(exc).__name__}: {exc}'

            rows.append(row)

    result = pd.DataFrame(rows)
    problems = result.loc[~result['exists'] | ~result['readable']]
    if problems.empty:
        print(f'Se validaron correctamente {len(result)} archivos.')
    else:
        print(
            f'Se detectaron problemas en {len(problems)} de {len(result)} archivos. '
            'Revise las columnas path y error.'
        )

    return result



## 2. Clasificador y validación cruzada

El preprocesamiento se ejecuta **dentro** de cada fold mediante un `Pipeline` (imputación, estandarización y SVM). Esto evita fuga de información.

El modo recomendado es `MODEL_POLICY='fixed'`: los hiperparámetros se fijan antes de mirar los resultados. El modo `saved_template` carga un modelo, extrae el estimador y lo clona; esto evita reutilizar el ajuste, pero **no corrige** el sesgo si sus hiperparámetros fueron elegidos con todo el mismo conjunto de datos. En ese caso, la selección de hiperparámetros debería repetirse dentro de cada fold y de cada permutación (validación anidada).


In [ ]:
# -----------------------------------------------------------------------------
# MODELO, VALIDACIÓN CRUZADA Y PUNTAJE
# -----------------------------------------------------------------------------
ANALYSIS_ALIASES = {
    'energy': ['energy'],
    'coactivation': ['coactivation', 'coactiv'],
    'entropy': ['entropy'],
    'kl_divergence': ['kl_divergence', 'kl_div', 'kldiv'],
    'hurst': ['hurst', 'h'],
}
CONTRAST_MODEL_LABEL = {
    'age_main': 'age',
    'condition_main': 'cond',
    'condition_at_3m': 'cond',
    'condition_at_6m': 'cond',
    'age_within_WT': 'age',
    'age_within_5xFAD': 'age',
}


def fixed_svm() -> Pipeline:
    return Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('svm', SVC(
            kernel=SVM_KERNEL,
            C=SVM_C,
            class_weight=SVM_CLASS_WEIGHT,
        )),
    ])


def _extract_estimator(loaded):
    if isinstance(loaded, dict):
        for key in ('model', 'estimator', 'classifier', 'svm'):
            if key in loaded:
                return loaded[key]
        raise KeyError(f'El archivo joblib contiene un dict sin estimator conocido: {list(loaded)}')
    return loaded


def find_saved_model(
    cfg: StimulusSetConfig,
    analysis: str,
    contrast: str,
    stimulus_id: str,
) -> Path | None:
    override = SAVED_MODEL_OVERRIDES.get((cfg.stimulus_type, analysis, contrast, stimulus_id))
    if override:
        path = PROJECT_ROOT / override
        if not path.exists():
            raise FileNotFoundError(path)
        return path

    if not cfg.model_dir.exists():
        return None

    label = CONTRAST_MODEL_LABEL.get(contrast, contrast)
    candidates: list[Path] = []
    for alias in ANALYSIS_ALIASES.get(analysis, [analysis]):
        bases = [
            f'{alias}_{label}_svm_{stimulus_id}',
            f'{alias}_{label}_{stimulus_id}',
            f'{alias}_{label}_svm',
            f'{alias}_{label}',
        ]
        for base in bases:
            for suffix in ('.pkl', '.joblib'):
                path = cfg.model_dir / f'{base}{suffix}'
                if path.exists():
                    candidates.append(path)
    unique = list(dict.fromkeys(candidates))
    if len(unique) > 1:
        warnings.warn(f'Múltiples modelos candidatos; se utilizará {unique[0]}: {unique}')
    return unique[0] if unique else None


def make_estimator(
    cfg: StimulusSetConfig,
    analysis: str,
    contrast: str,
    stimulus_id: str,
):
    if MODEL_POLICY == 'fixed':
        return fixed_svm(), 'fixed_svm'
    if MODEL_POLICY == 'saved_template':
        path = find_saved_model(cfg, analysis, contrast, stimulus_id)
        if path is None:
            warnings.warn(
                f'No se encontró modelo para {cfg.stimulus_type}/{analysis}/{contrast}/{stimulus_id}; '
                'se usará el SVM fijo.'
            )
            return fixed_svm(), 'fixed_svm_fallback'
        estimator = _extract_estimator(joblib.load(path))
        return clone(estimator), str(path)
    raise ValueError(f'MODEL_POLICY no reconocido: {MODEL_POLICY}')


def make_cv_splits(y: np.ndarray, seed: int) -> list[tuple[np.ndarray, np.ndarray]]:
    y = np.asarray(y)
    if CV_SCHEME == 'loo':
        return list(LeaveOneOut().split(np.zeros((len(y), 1)), y))
    if CV_SCHEME == 'stratified_kfold':
        counts = pd.Series(y).value_counts()
        n_splits = min(N_SPLITS, int(counts.min()))
        if n_splits < 2:
            raise ValueError(f'No hay suficientes individuos por clase: {counts.to_dict()}')
        cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        return list(cv.split(np.zeros((len(y), 1)), y))
    raise ValueError(f'CV_SCHEME no reconocido: {CV_SCHEME}')


def cv_scores(estimator, X: np.ndarray, y: np.ndarray, seed: int) -> dict[str, float]:
    splits = make_cv_splits(y, seed)
    predictions = cross_val_predict(
        clone(estimator), X, y, cv=splits, method='predict', n_jobs=1,
    )
    return {
        'balanced_accuracy': balanced_accuracy_score(y, predictions),
        'accuracy': accuracy_score(y, predictions),
    }


def binomial_accuracy_threshold(n: int, n_classes: int = 2, alpha: float = 0.05) -> float:
    """Umbral diagnóstico. No sustituye la permutación con CV."""
    chance = 1 / n_classes
    # Convención de la tabla del paper: cuantil binomial 1-alpha.
    k = int(binom.ppf(1 - alpha, n, chance))
    return min(k / n, 1.0)


In [ ]:
# -----------------------------------------------------------------------------
# TAMAÑOS DE EFECTO
# -----------------------------------------------------------------------------
def cohen_d(x_negative: np.ndarray, x_positive: np.ndarray) -> float:
    x0 = np.asarray(x_negative, dtype=float)
    x1 = np.asarray(x_positive, dtype=float)
    n0, n1 = len(x0), len(x1)
    if n0 < 2 or n1 < 2:
        return np.nan
    pooled_var = ((n0 - 1) * np.var(x0, ddof=1) + (n1 - 1) * np.var(x1, ddof=1)) / (n0 + n1 - 2)
    if pooled_var <= 0:
        return np.nan
    return (np.mean(x1) - np.mean(x0)) / np.sqrt(pooled_var)


def hedges_g_from_d(d: float, n0: int, n1: int) -> float:
    if not np.isfinite(d):
        return np.nan
    df = n0 + n1 - 2
    if df <= 1:
        return np.nan
    correction = 1 - 3 / (4 * df - 1)
    return correction * d


def cliffs_delta(x_negative: np.ndarray, x_positive: np.ndarray) -> float:
    """Positivo cuando la clase positiva tiende a valores mayores."""
    x0 = np.asarray(x_negative, dtype=float)
    x1 = np.asarray(x_positive, dtype=float)
    comparisons = np.sign(x1[:, None] - x0[None, :])
    return float(comparisons.mean())


def regularized_mahalanobis(X_negative: np.ndarray, X_positive: np.ndarray) -> float:
    X0 = np.asarray(X_negative, dtype=float)
    X1 = np.asarray(X_positive, dtype=float)
    if X0.ndim == 1:
        X0 = X0[:, None]
    if X1.ndim == 1:
        X1 = X1[:, None]
    if len(X0) < 2 or len(X1) < 2:
        return np.nan
    diff = X1.mean(axis=0) - X0.mean(axis=0)
    residuals = np.vstack([X0 - X0.mean(axis=0), X1 - X1.mean(axis=0)])
    covariance = LedoitWolf().fit(residuals).covariance_
    value = float(diff @ np.linalg.pinv(covariance) @ diff)
    return float(np.sqrt(max(value, 0.0)))


def bootstrap_effect_ci(
    x_negative: np.ndarray,
    x_positive: np.ndarray,
    statistic: Callable[[np.ndarray, np.ndarray], float],
    n_bootstrap: int,
    seed: int,
) -> tuple[float, float]:
    rng = np.random.default_rng(seed)
    x0 = np.asarray(x_negative, dtype=float)
    x1 = np.asarray(x_positive, dtype=float)
    values = np.empty(n_bootstrap, dtype=float)
    for i in range(n_bootstrap):
        b0 = rng.choice(x0, size=len(x0), replace=True)
        b1 = rng.choice(x1, size=len(x1), replace=True)
        values[i] = statistic(b0, b1)
    values = values[np.isfinite(values)]
    if not len(values):
        return np.nan, np.nan
    return tuple(np.quantile(values, [0.025, 0.975]))


def effect_size_rows(
    frame: pd.DataFrame,
    feature_group: FeatureGroup,
    contrast: Contrast,
    context: Mapping[str, object],
    seed: int,
) -> list[dict]:
    rows = []
    negative_mask = frame[contrast.target] == contrast.negative_class
    positive_mask = frame[contrast.target] == contrast.positive_class

    for feature_column in feature_group.columns:
        x0 = frame.loc[negative_mask, feature_column].to_numpy(dtype=float)
        x1 = frame.loc[positive_mask, feature_column].to_numpy(dtype=float)
        d = cohen_d(x0, x1)
        g = hedges_g_from_d(d, len(x0), len(x1))
        d_low, d_high = bootstrap_effect_ci(x0, x1, cohen_d, N_BOOTSTRAP_EFFECT, seed)

        def g_stat(a, b):
            return hedges_g_from_d(cohen_d(a, b), len(a), len(b))

        g_low, g_high = bootstrap_effect_ci(x0, x1, g_stat, N_BOOTSTRAP_EFFECT, seed + 1)
        rows.append({
            **context,
            'feature_column': feature_column,
            'negative_class': contrast.negative_class,
            'positive_class': contrast.positive_class,
            'n_negative': len(x0),
            'n_positive': len(x1),
            'mean_negative': float(np.mean(x0)),
            'mean_positive': float(np.mean(x1)),
            'sd_negative': float(np.std(x0, ddof=1)) if len(x0) > 1 else np.nan,
            'sd_positive': float(np.std(x1, ddof=1)) if len(x1) > 1 else np.nan,
            'cohen_d': d,
            'cohen_d_ci_low': d_low,
            'cohen_d_ci_high': d_high,
            'hedges_g': g,
            'hedges_g_ci_low': g_low,
            'hedges_g_ci_high': g_high,
            'cliffs_delta': cliffs_delta(x0, x1),
        })
    return rows


## 3. Prueba de permutación sincronizada y maxT

Para cada combinación `tipo de estímulo × metodología × contraste`:

1. Se utiliza el mismo conjunto completo de ratones para todos sus estímulos.
2. Se calcula la exactitud balanceada observada con todo el pipeline de CV.
3. En cada permutación se intercambia la etiqueta objetivo dentro de los bloques del factor de control.
4. Se vuelve a entrenar y evaluar el SVM para cada estímulo.
5. El p-valor usa la corrección de Monte Carlo `(b + 1)/(B + 1)`, por lo que nunca es cero.
6. La distribución del máximo entre estímulos entrega `p_maxT_fwer`, que controla el error familiar al escoger el mejor estímulo.
7. Se calculan dos pruebas globales por metodología:
   - `global_any_p`: evidencia de que al menos un estímulo separa (máximo).
   - `global_consistency_p`: evidencia de desempeño promedio entre estímulos (media).


In [ ]:
# -----------------------------------------------------------------------------
# PERMUTACIONES RESTRINGIDAS Y ANÁLISIS DE UNA FAMILIA DE ESTÍMULOS
# -----------------------------------------------------------------------------
def restricted_permutation(
    y: np.ndarray,
    blocks: np.ndarray | None,
    rng: np.random.Generator,
) -> np.ndarray:
    y = np.asarray(y)
    if blocks is None:
        return rng.permutation(y)
    blocks = np.asarray(blocks)
    result = y.copy()
    for block in pd.unique(blocks):
        index = np.flatnonzero(blocks == block)
        result[index] = rng.permutation(y[index])
    return result


def conservative_quantile(values: np.ndarray, q: float) -> float:
    values = np.asarray(values, dtype=float)
    try:
        return float(np.quantile(values, q, method='higher'))
    except TypeError:  # NumPy antiguo
        return float(np.quantile(values, q, interpolation='higher'))


def monte_carlo_p(null_values: np.ndarray, observed: float) -> float:
    null_values = np.asarray(null_values, dtype=float)
    valid = null_values[np.isfinite(null_values)]
    return (1 + int(np.sum(valid >= observed))) / (len(valid) + 1)


def _one_permutation_scores(
    seed: int,
    y: np.ndarray,
    blocks: np.ndarray | None,
    X_list: Sequence[np.ndarray],
    estimators: Sequence,
) -> np.ndarray:
    rng = np.random.default_rng(seed)
    y_perm = restricted_permutation(y, blocks, rng)
    scores = np.empty(len(X_list), dtype=float)
    for j, (X, estimator) in enumerate(zip(X_list, estimators)):
        scores[j] = cv_scores(estimator, X, y_perm, RANDOM_STATE)['balanced_accuracy']
    return scores


def analyze_method_family(
    cfg: StimulusSetConfig,
    analysis: str,
    filename: str,
    contrast: Contrast,
) -> tuple[pd.DataFrame, pd.DataFrame, dict, dict, list[dict]]:
    path = cfg.base_dir / filename
    df = read_feature_table(path)
    groups = discover_feature_groups(df, cfg.stimulus_type, analysis, cfg.expected_stimuli)
    contrasted = apply_contrast(df, contrast)

    all_feature_columns = sorted({column for group in groups for column in group.columns})
    numeric = contrasted[all_feature_columns].apply(pd.to_numeric, errors='coerce')
    complete_mask = numeric.notna().all(axis=1)
    frame = pd.concat([
        contrasted.loc[complete_mask, ['subject_id', 'age', 'condition']].reset_index(drop=True),
        numeric.loc[complete_mask].reset_index(drop=True),
    ], axis=1)

    if len(frame) < 4:
        raise ValueError(f'Muy pocos casos completos para {cfg.stimulus_type}/{analysis}/{contrast.name}.')

    class_counts = frame[contrast.target].value_counts()
    if set(class_counts.index) != {contrast.negative_class, contrast.positive_class}:
        raise ValueError(
            f'Clases incompletas en {cfg.stimulus_type}/{analysis}/{contrast.name}: '
            f'{class_counts.to_dict()}'
        )
    if int(class_counts.min()) < 2:
        raise ValueError(f'Se requieren al menos 2 individuos por clase: {class_counts.to_dict()}')

    y = frame[contrast.target].to_numpy()
    blocks = frame[contrast.nuisance].to_numpy() if contrast.nuisance else None

    if contrast.nuisance:
        contingency = pd.crosstab(frame[contrast.nuisance], frame[contrast.target])
        missing_within_block = [
            block for block, row in contingency.iterrows()
            if any(row.get(label, 0) == 0 for label in (contrast.negative_class, contrast.positive_class))
        ]
        if missing_within_block:
            raise ValueError(
                f'No es posible permutar {contrast.target} dentro de {contrast.nuisance}: '
                f'los bloques {missing_within_block} no contienen ambas clases. '
                f'Tabla: {contingency.to_dict()}'
            )

    X_list: list[np.ndarray] = []
    estimators = []
    estimator_sources = []
    for group in groups:
        X_list.append(frame[list(group.columns)].to_numpy(dtype=float))
        estimator, source = make_estimator(cfg, analysis, contrast.name, group.stimulus_id)
        estimators.append(estimator)
        estimator_sources.append(source)

    observed_balanced = np.empty(len(groups), dtype=float)
    observed_accuracy = np.empty(len(groups), dtype=float)
    for j, (X, estimator) in enumerate(zip(X_list, estimators)):
        scores = cv_scores(estimator, X, y, RANDOM_STATE)
        observed_balanced[j] = scores['balanced_accuracy']
        observed_accuracy[j] = scores['accuracy']

    seed_sequence = np.random.SeedSequence(RANDOM_STATE).spawn(N_PERMUTATIONS)
    permutation_seeds = [int(sequence.generate_state(1)[0]) for sequence in seed_sequence]
    permutation_rows = Parallel(n_jobs=N_JOBS, verbose=0)(
        delayed(_one_permutation_scores)(seed, y, blocks, X_list, estimators)
        for seed in permutation_seeds
    )
    null_matrix = np.vstack(permutation_rows)
    max_null = np.nanmax(null_matrix, axis=1)
    mean_null = np.nanmean(null_matrix, axis=1)

    max_threshold = conservative_quantile(max_null, 1 - ALPHA)
    raw_rows = []
    effect_rows_all: list[dict] = []
    for j, group in enumerate(groups):
        raw_p = monte_carlo_p(null_matrix[:, j], observed_balanced[j])
        max_t_p = monte_carlo_p(max_null, observed_balanced[j])
        context = {
            'stimulus_type': cfg.stimulus_type,
            'analysis': analysis,
            'contrast': contrast.name,
            'stimulus_id': group.stimulus_id,
        }
        effect_rows = effect_size_rows(
            frame, group, contrast, context,
            seed=RANDOM_STATE + j * 17,
        )
        effect_rows_all.extend(effect_rows)
        hedges_values = np.array([row['hedges_g'] for row in effect_rows], dtype=float)

        negative = frame[contrast.target] == contrast.negative_class
        positive = frame[contrast.target] == contrast.positive_class
        mahalanobis = regularized_mahalanobis(
            frame.loc[negative, list(group.columns)].to_numpy(dtype=float),
            frame.loc[positive, list(group.columns)].to_numpy(dtype=float),
        )
        raw_rows.append({
            **context,
            'feature_columns': ', '.join(group.columns),
            'n_features': len(group.columns),
            'n_total': len(frame),
            'n_negative': int((y == contrast.negative_class).sum()),
            'n_positive': int((y == contrast.positive_class).sum()),
            'negative_class': contrast.negative_class,
            'positive_class': contrast.positive_class,
            'nuisance_factor': contrast.nuisance or '',
            'cv_scheme': CV_SCHEME,
            'model_source': estimator_sources[j],
            'balanced_accuracy': observed_balanced[j],
            'accuracy': observed_accuracy[j],
            'theoretical_chance': 0.5,
            'binomial_threshold_diagnostic': binomial_accuracy_threshold(len(frame), 2, ALPHA),
            'permutation_threshold_raw': conservative_quantile(null_matrix[:, j], 1 - ALPHA),
            'permutation_threshold_maxT': max_threshold,
            'p_raw': raw_p,
            'p_maxT_fwer_within_analysis': max_t_p,
            'significant_raw': raw_p <= ALPHA,
            'significant_maxT': max_t_p <= ALPHA,
            'max_abs_hedges_g': float(np.nanmax(np.abs(hedges_values))),
            'mean_abs_hedges_g': float(np.nanmean(np.abs(hedges_values))),
            'mahalanobis_distance': mahalanobis,
        })

    tests_df = pd.DataFrame(raw_rows)
    effects_df = pd.DataFrame(effect_rows_all)
    summary = {
        'stimulus_type': cfg.stimulus_type,
        'analysis': analysis,
        'contrast': contrast.name,
        'cv_scheme': CV_SCHEME,
        'n_total': len(frame),
        'n_stimuli': len(groups),
        'best_stimulus': groups[int(np.argmax(observed_balanced))].stimulus_id,
        'best_balanced_accuracy': float(np.max(observed_balanced)),
        'mean_balanced_accuracy': float(np.mean(observed_balanced)),
        'global_any_statistic_max': float(np.max(observed_balanced)),
        'global_any_p': monte_carlo_p(max_null, float(np.max(observed_balanced))),
        'global_consistency_statistic_mean': float(np.mean(observed_balanced)),
        'global_consistency_p': monte_carlo_p(mean_null, float(np.mean(observed_balanced))),
    }
    quality = {
        'stimulus_type': cfg.stimulus_type,
        'analysis': analysis,
        'contrast': contrast.name,
        'source_file': str(path),
        'n_original': len(contrasted),
        'n_complete_common': len(frame),
        'n_removed_missing': int((~complete_mask).sum()),
        'class_counts': json.dumps(class_counts.to_dict(), ensure_ascii=False),
        'n_stimuli_detected': len(groups),
        'expected_stimuli': cfg.expected_stimuli,
    }
    null_payload = {
        'key': f'{cfg.stimulus_type}__{analysis}__{contrast.name}',
        'null_matrix': null_matrix,
        'stimulus_ids': np.array([g.stimulus_id for g in groups], dtype=object),
    }
    return tests_df, effects_df, summary, null_payload, [quality]


In [ ]:
# -----------------------------------------------------------------------------
# EJECUCIÓN COMPLETA, FDR Y RANKING
# -----------------------------------------------------------------------------
def apply_multiple_testing(results: pd.DataFrame) -> pd.DataFrame:
    out = results.copy()
    out['p_fdr_bh_primary_family'] = np.nan
    out['sig_fdr_bh_primary_family'] = False
    out['p_fdr_by_primary_family'] = np.nan
    out['sig_fdr_by_primary_family'] = False

    # Familia principal: misma pregunta biológica y mismo tipo de estímulo.
    family_columns = ['stimulus_type', 'contrast', 'cv_scheme']
    for _, index in out.groupby(family_columns, sort=False).groups.items():
        index = list(index)
        p = out.loc[index, 'p_raw'].to_numpy(dtype=float)
        reject_bh, p_bh, _, _ = multipletests(p, alpha=ALPHA, method='fdr_bh')
        reject_by, p_by, _, _ = multipletests(p, alpha=ALPHA, method='fdr_by')
        out.loc[index, 'p_fdr_bh_primary_family'] = p_bh
        out.loc[index, 'sig_fdr_bh_primary_family'] = reject_bh
        out.loc[index, 'p_fdr_by_primary_family'] = p_by
        out.loc[index, 'sig_fdr_by_primary_family'] = reject_by

    # FDR exploratorio dentro de metodología; maxT es la corrección principal
    # para seleccionar estímulos dentro de una metodología.
    out['p_fdr_bh_within_analysis'] = np.nan
    out['sig_fdr_bh_within_analysis'] = False
    within_columns = ['stimulus_type', 'contrast', 'analysis', 'cv_scheme']
    for _, index in out.groupby(within_columns, sort=False).groups.items():
        index = list(index)
        p = out.loc[index, 'p_raw'].to_numpy(dtype=float)
        reject, p_adj, _, _ = multipletests(p, alpha=ALPHA, method='fdr_bh')
        out.loc[index, 'p_fdr_bh_within_analysis'] = p_adj
        out.loc[index, 'sig_fdr_bh_within_analysis'] = reject
    return out


def run_all_analyses():
    tests_frames = []
    effects_frames = []
    summary_rows = []
    quality_rows = []
    null_payloads = []

    for cfg in STIMULUS_SETS:
        if SELECTED_STIMULUS_TYPES is not None and cfg.stimulus_type not in SELECTED_STIMULUS_TYPES:
            continue
        for analysis, filename in cfg.files.items():
            if SELECTED_ANALYSES is not None and analysis not in SELECTED_ANALYSES:
                continue
            for contrast in CONTRASTS:
                if SELECTED_CONTRASTS is not None and contrast.name not in SELECTED_CONTRASTS:
                    continue
                print(f'Analizando {cfg.stimulus_type} | {analysis} | {contrast.name}')
                tests, effects, summary, null_payload, quality = analyze_method_family(
                    cfg, analysis, filename, contrast,
                )
                tests_frames.append(tests)
                effects_frames.append(effects)
                summary_rows.append(summary)
                quality_rows.extend(quality)
                null_payloads.append(null_payload)

    results = apply_multiple_testing(pd.concat(tests_frames, ignore_index=True))
    effects = pd.concat(effects_frames, ignore_index=True)
    method_summary = pd.DataFrame(summary_rows)
    quality = pd.DataFrame(quality_rows)

    # Corrección FDR de las pruebas globales de metodologías dentro de cada
    # tipo de estímulo/contraste.
    for p_column in ['global_any_p', 'global_consistency_p']:
        adjusted_column = f'{p_column}_fdr_bh'
        significant_column = f'{p_column}_sig_fdr_bh'
        method_summary[adjusted_column] = np.nan
        method_summary[significant_column] = False
        for _, index in method_summary.groupby(
            ['stimulus_type', 'contrast', 'cv_scheme'], sort=False
        ).groups.items():
            index = list(index)
            reject, adjusted, _, _ = multipletests(
                method_summary.loc[index, p_column].to_numpy(dtype=float),
                alpha=ALPHA, method='fdr_bh',
            )
            method_summary.loc[index, adjusted_column] = adjusted
            method_summary.loc[index, significant_column] = reject

    ranking = results.sort_values(
        [
            'stimulus_type', 'contrast',
            'p_maxT_fwer_within_analysis',
            'p_fdr_bh_primary_family',
            'balanced_accuracy',
            'max_abs_hedges_g',
        ],
        ascending=[True, True, True, True, False, False],
    ).reset_index(drop=True)
    ranking['rank_within_question'] = (
        ranking.groupby(['stimulus_type', 'contrast']).cumcount() + 1
    )
    return results, effects, method_summary, quality, ranking, null_payloads


In [ ]:
# -----------------------------------------------------------------------------
# EXPORTACIÓN A EXCEL
# -----------------------------------------------------------------------------
def _excel_safe_frame(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for column in out.columns:
        if out[column].map(lambda x: isinstance(x, (dict, list, tuple, np.ndarray))).any():
            out[column] = out[column].map(
                lambda x: json.dumps(x, ensure_ascii=False, default=str)
                if isinstance(x, (dict, list, tuple, np.ndarray)) else x
            )
    return out


def export_excel(
    path: Path,
    results: pd.DataFrame,
    effects: pd.DataFrame,
    method_summary: pd.DataFrame,
    quality: pd.DataFrame,
    ranking: pd.DataFrame,
) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    notes = pd.DataFrame({
        'Campo': [
            'p_raw',
            'p_maxT_fwer_within_analysis',
            'p_fdr_bh_primary_family',
            'p_fdr_by_primary_family',
            'global_any_p',
            'global_consistency_p',
            'Cohen d / Hedges g',
            'Signo del efecto',
        ],
        'Interpretación': [
            'p de permutación para un estímulo y metodología concretos.',
            'FWER maxT al seleccionar entre estímulos de la misma metodología.',
            'FDR BH entre todas las metodologías y estímulos de una pregunta biológica.',
            'FDR BY conservador ante dependencia arbitraria.',
            'Prueba global: existe al menos un estímulo separador para la metodología.',
            'Prueba global: la metodología muestra separación promedio entre estímulos.',
            'Tamaño de efecto descriptivo; Hedges g corrige el sesgo de Cohen d con n pequeño.',
            'Positivo = media de la clase positiva mayor que la clase negativa.',
        ],
    })
    config = pd.DataFrame([
        {'parameter': 'N_PERMUTATIONS', 'value': N_PERMUTATIONS},
        {'parameter': 'RANDOM_STATE', 'value': RANDOM_STATE},
        {'parameter': 'ALPHA', 'value': ALPHA},
        {'parameter': 'CV_SCHEME', 'value': CV_SCHEME},
        {'parameter': 'N_SPLITS', 'value': N_SPLITS},
        {'parameter': 'MODEL_POLICY', 'value': MODEL_POLICY},
        {'parameter': 'SVM_KERNEL', 'value': SVM_KERNEL},
        {'parameter': 'SVM_C', 'value': SVM_C},
        {'parameter': 'INCLUDE_SIMPLE_EFFECTS', 'value': INCLUDE_SIMPLE_EFFECTS},
        {'parameter': 'SELECTED_STIMULUS_TYPES', 'value': str(SELECTED_STIMULUS_TYPES)},
        {'parameter': 'SELECTED_ANALYSES', 'value': str(SELECTED_ANALYSES)},
        {'parameter': 'SELECTED_CONTRASTS', 'value': str(SELECTED_CONTRASTS)},
    ])

    with pd.ExcelWriter(path, engine='xlsxwriter') as writer:
        sheets: dict[str, pd.DataFrame] = {
            'All_tests': results,
            'Method_summary': method_summary,
            'Best_candidates': ranking,
            'Effect_sizes': effects,
            'Data_quality': quality,
            'Configuration': config,
            'Method_notes': notes,
        }
        for stimulus_type, sheet_name in {
            '1D_deterministic': '1D_det',
            '1D_stochastic': '1D_stoch',
            '2D_stochastic': '2D_stoch',
        }.items():
            sheets[sheet_name] = results.loc[results['stimulus_type'] == stimulus_type]

        workbook = writer.book
        header_format = workbook.add_format({
            'bold': True, 'font_color': 'white', 'bg_color': '#1F4E78',
            'border': 1, 'align': 'center', 'valign': 'vcenter',
        })
        percent_format = workbook.add_format({'num_format': '0.0%'})
        p_format = workbook.add_format({'num_format': '0.0000'})
        wrap_format = workbook.add_format({'text_wrap': True, 'valign': 'top'})

        for sheet_name, frame in sheets.items():
            frame = _excel_safe_frame(frame)
            frame.to_excel(writer, sheet_name=sheet_name, index=False)
            worksheet = writer.sheets[sheet_name]
            worksheet.freeze_panes(1, 0)
            worksheet.autofilter(0, 0, max(len(frame), 1), max(len(frame.columns) - 1, 0))
            worksheet.set_row(0, 30, header_format)

            for col_idx, column in enumerate(frame.columns):
                values = frame[column].astype(str) if len(frame) else pd.Series(dtype=str)
                max_len = max([len(str(column))] + values.head(500).map(len).tolist())
                width = min(max(max_len + 2, 11), 38)
                fmt = wrap_format if width >= 30 else None
                if any(token in column.lower() for token in ('accuracy', 'threshold', 'chance')):
                    fmt = percent_format
                elif column.lower().startswith('p_') or column.lower().endswith('_p'):
                    fmt = p_format
                worksheet.set_column(col_idx, col_idx, width, fmt)

            # Resalta significancia y desempeño alto cuando las columnas existen.
            if len(frame):
                for p_column in [
                    'p_maxT_fwer_within_analysis',
                    'p_fdr_bh_primary_family',
                    'global_any_p_fdr_bh',
                ]:
                    if p_column in frame.columns:
                        col = frame.columns.get_loc(p_column)
                        worksheet.conditional_format(
                            1, col, len(frame), col,
                            {'type': 'cell', 'criteria': '<=', 'value': ALPHA,
                             'format': workbook.add_format({'bg_color': '#C6EFCE', 'font_color': '#006100'})}
                        )
                if 'balanced_accuracy' in frame.columns:
                    col = frame.columns.get_loc('balanced_accuracy')
                    worksheet.conditional_format(
                        1, col, len(frame), col,
                        {'type': '3_color_scale', 'min_color': '#F8696B',
                         'mid_color': '#FFEB84', 'max_color': '#63BE7B'}
                    )

    print(f'Excel guardado en: {path.resolve()}')


## 4. Validación de archivos y ejecución

Primero ejecute la celda de validación. Debe detectar 21 estímulos para 1D determinista y 3 para cada familia estocástica. Si Hurst 2D no se agrupa correctamente, complete `FEATURE_GROUP_OVERRIDES` en la configuración.

La ejecución final puede ser costosa. Para una prueba rápida use `N_PERMUTATIONS=99`; para resultados reportables use al menos `9_999`. Con muchos SVM y LOO, se recomienda ejecutar por tipo de estímulo o usar `stratified_kfold` como análisis principal y LOO como sensibilidad.


In [ ]:
file_status = validate_project_files(STIMULUS_SETS)
file_status


In [ ]:
# Vista previa de las columnas/estímulos detectados antes de ejecutar permutaciones.
discovery_rows = []
for cfg in STIMULUS_SETS:
    for analysis, filename in cfg.files.items():
        path = cfg.base_dir / filename
        if not path.exists():
            continue
        df = read_feature_table(path)
        groups = discover_feature_groups(df, cfg.stimulus_type, analysis, cfg.expected_stimuli)
        for group in groups:
            discovery_rows.append({
                'stimulus_type': cfg.stimulus_type,
                'analysis': analysis,
                'stimulus_id': group.stimulus_id,
                'feature_columns': list(group.columns),
            })
feature_discovery = pd.DataFrame(discovery_rows)
feature_discovery


In [ ]:
# EJECUCIÓN PRINCIPAL
if not file_status['exists'].all():
    raise FileNotFoundError(
        'Faltan archivos de entrada. Revise PROJECT_ROOT y la tabla file_status antes de ejecutar.'
    )

(
    results_df,
    effect_sizes_df,
    method_summary_df,
    data_quality_df,
    ranking_df,
    null_payloads,
) = run_all_analyses()

excel_path = OUTPUT_DIR / 'neuroscience_significance_results.xlsx'
export_excel(
    excel_path,
    results_df,
    effect_sizes_df,
    method_summary_df,
    data_quality_df,
    ranking_df,
)

if SAVE_NULL_DISTRIBUTIONS:
    arrays = {}
    for payload in null_payloads:
        key = payload['key']
        arrays[f'{key}__null'] = payload['null_matrix']
        arrays[f'{key}__stimulus_ids'] = payload['stimulus_ids']
    np.savez_compressed(OUTPUT_DIR / 'permutation_null_distributions.npz', **arrays)

print(f'P mínimo resoluble: {1 / (N_PERMUTATIONS + 1):.6g}')


In [ ]:
# RESULTADOS PRINCIPALES
significant_candidates = results_df.loc[
    results_df['sig_fdr_bh_primary_family']
    | results_df['significant_maxT']
].sort_values([
    'stimulus_type', 'contrast',
    'p_maxT_fwer_within_analysis',
    'p_fdr_bh_primary_family',
    'balanced_accuracy',
], ascending=[True, True, True, True, False])

print('Pruebas individuales significativas tras al menos una corrección principal:')
display(significant_candidates)

print('Resumen global por metodología:')
display(method_summary_df.sort_values([
    'stimulus_type', 'contrast', 'global_any_p_fdr_bh', 'best_balanced_accuracy'
], ascending=[True, True, True, False]))


## Interpretación recomendada

- **No interpretar solo la exactitud observada.** Con 30–40 individuos pueden aparecer exactitudes altas por azar; el p de permutación es la referencia principal.
- Para afirmar que un **estímulo concreto** es candidato, priorice `p_maxT_fwer_within_analysis ≤ 0.05` y revise además el tamaño de efecto y su intervalo.
- Para afirmar que una **metodología** sirve como separador, use `global_any_p_fdr_bh` (al menos un estímulo) o `global_consistency_p_fdr_bh` (consistencia promedio).
- `p_fdr_bh_primary_family` responde a la selección exploratoria entre todos los pares metodología–estímulo de una pregunta biológica. `p_fdr_by_primary_family` es una alternativa más conservadora.
- Cohen *d* es descriptivo; con estas muestras se recomienda reportar también **Hedges g**, que corrige el sesgo de muestra pequeña. Para Hx/Hy se reportan efectos por eje y una distancia de Mahalanobis regularizada.
- Un resultado pooled de genotipo o edad no reemplaza el estudio de una interacción. Si hay una hipótesis de interacción, active `INCLUDE_SIMPLE_EFFECTS` o modele explícitamente el diseño factorial.
- Si los hiperparámetros de los SVM se eligieron mirando estas mismas observaciones, el análisis confirmatorio debe repetir esa selección dentro de cada fold y permutación.
